In [ ]:
from pyspark.sql.functions import col, isnull, when
from pyspark.sql.types import TimestampType
from datetime import date, timedelta

In [1]:
# Remove this cell before running Data Factory Pipeline
start_date = date.today() - timedelta(7)

StatementMeta(, , -1, SessionStarting, , SessionStarting, True)

In [ ]:
# Load JSON data i nto a Spark DataFrame
df = spark.read.option("multiline", "true").json(f"Files/{start_date}_earthquake_data.json")

In [ ]:
df.head()

In [ ]:
# Reshape earthquake data
df = (
    df
    .select(
        "id",
        col("geometry.coordinates").getItem(0).alias("longitude"),
        col("geometry.coordinates").getItem(1).alias("latitude"),
        col("geometry.coordinates").getItem(2).alias("elevation"),
        col("properties.title").alias("title"),
        col("properties.place").alias("place_description"),
        col("properties.sig").alias("sig"),
        col("properties.mag").alias("mag"),
        col("properties.magType").alias("magType"),
        col("properties.time").alias("time"),
        col("properties.updated").alias("updated")
    )
)

In [ ]:
# Validate data: check for misisng or null values
df = (
    df
    .withColumn("longitude", when(isnull(col("longitude")), 0).otherwise(col("longitude")))
    .withColumn("latitude", when(isnull(col("latitude")), 0).otherwise(col("latitude")))
    .withColumn("time", when(isnull(col("time")), 0).otherwise(col("time")))
)

In [ ]:
# Convert "time" and "updated" to timestamp
df = (
    df
    .withColumn("time", (col("time") / 1000).cast(TimestampType()))
    .withColumn("updated", (col("updated") / 1000).cast(TimestampType()))
)

In [ ]:
# Append to silver table 
df.write.mode("append").saveAsTable("earthquake_events_silver")